# ELI test
look at relationship with SST, d/dt(SST)

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import pandas as pd

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_ddt(data):
    """differentiate with respect to time"""
    data_ = copy.deepcopy(data)
    data_ = data_.assign_coords({"t_idx": ("time", np.arange(len(data_.time)))})
    data_ = data_.swap_dims({"time": "t_idx"})

    ## differentiate
    ddt_data = data_.differentiate("t_idx").swap_dims({"t_idx": "time"})

    ## mult. by 12 to convert from 1/mo to 1/yr
    return 12 * ddt_data

## Load data

### T, h

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True, load_h_cust=True, max_grad=True)

#### Load ELI

In [ ]:
## Load ELI
eli_all = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/eli.nc"))
eli_forced, eli = src.utils.separate_forced(eli_all)

## add to data
Th = xr.merge([Th, eli])

### preprocess

In [ ]:
## standardize (for convenience)
Th /= Th.std()

## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

## drop last year because of NaNs
Th_rolling = Th_rolling.sel(year=slice(None, 2080))

## compute grad
Th_rolling["dTdx"] = Th_rolling["T_3"] - Th_rolling["T_4"]

#### eli

In [ ]:
## load dT/dx data
# dTdx = xr.open_dataarray(pathlib.Path(SAVE_FP, "cesm_dTdx.nc"))
dTdx = xr.open_dataarray(pathlib.Path(SAVE_FP, "cesm_dTdx_Tw.nc"))

## scale by initial value
dTdx_scale = dTdx / dTdx.isel(year=0).mean(["month"])

## update year
dTdx_scale = dTdx_scale.assign_coords({"year": Th_rolling.year})

for v in list(Th_rolling):
    if "eli" in v:
        Th_rolling[f"{v}_scaled"] = Th_rolling[v].groupby("time.month") * dTdx_scale

        ## subtrack median
        grouped = Th_rolling[f"{v}_scaled"].groupby("time.month")
        Th_rolling[f"{v}_scaled_median"] = grouped - grouped.median(["member", "time"])

In [ ]:
for v in ["T_3", "eli_05_scaled"]:
    Th_rolling[f"ddt_{v}"] = get_ddt(Th_rolling[v])

## Scatter plots

In [ ]:
import scipy.optimize


def psi_(x, a, b, c):
    """base transfer function"""
    return c * np.exp(a * x) + b


def get_psi(x, y):
    """get transfer function from data"""

    ## prep data
    stack = lambda x: x.stack(s=["time", "member"]).values
    x_ = stack(x)
    y_ = stack(y)

    ## get parameter fit
    p, _ = scipy.optimize.curve_fit(
        f=psi_,
        xdata=x_,
        ydata=y_,
        p0=np.array([1, -1, -1]),
        # po
        maxfev=2000,
    )

    ## define transfer func
    psi = lambda x: psi_(x, a=p[0], b=p[1], c=p[2])

    ## put parameters in array
    params = xr.DataArray(p, coords=dict(param=["a", "b", "c"])).rename("params")

    return psi, params


def get_psi_wrapper(xy, x_var, y_var, get_psi_func=get_psi):
    """get transfer function from data"""

    ## get transfer func
    psi_func, p = get_psi_func(x=xy[x_var], y=xy[y_var])

    ## get coefficient array (standardized units)
    z = np.arange(-1, 3.1, 0.1)
    z = xr.DataArray(z, coords=dict(sigma=z))

    ## evaluate function
    psi_eval = xr.zeros_like(z)
    psi_eval.values = psi_func(z.values)

    ## merge eval and params
    return xr.merge([psi_eval.rename("psi_eval"), p])


def psi_poly(T, c, b, a, e):
    """base transfer function"""
    # return e * x**3 + c * x**2 + b * x + a
    return (c / 3) * T**3 + (b / 2) * T**2 + a * T + e


def get_psi_poly(x, y):
    """get transfer function from data"""

    ## prep data
    stack = lambda x: x.stack(s=["time", "member"]).values
    x_ = stack(x)
    y_ = stack(y)

    ## get parameter fit
    p, _ = scipy.optimize.curve_fit(
        f=psi_poly,
        xdata=x_,
        ydata=y_,
    )

    ## define transfer func
    psi = lambda T: psi_poly(T, c=p[0], b=p[1], a=p[2], e=p[3])

    ## put params in dataarray
    params = xr.DataArray(p, coords=dict(param=["c", "b", "a", "e"])).rename("params")

    return psi, params

In [ ]:
## resample
X = Th_rolling.resample({"time": "QS-JAN"}).mean()

## subset for specific month
t_idx = dict(time=X.time.dt.month == 1)
X_ = X.sel(t_idx)

## plot years
PLOT_YRS = [1870, 2010, 2080]


## empty array to hold polyfit
res = []

fig, axs = plt.subplots(1, 3, figsize=(7.5, 2.5), layout="constrained")

for ax, yr in zip(axs, PLOT_YRS):

    res.append(
        get_psi_wrapper(
            xy=X_.sel(year=yr),
            x_var="eli_05_scaled",
            y_var="T_3",
            get_psi_func=get_psi_poly,
        )
    )

    ax.scatter(X_["eli_05_scaled"].sel(year=yr), X_["T_3"].sel(year=yr), s=10)
    ax.plot(res[-1].sigma, res[-1]["psi_eval"], c="k")

src.utils.set_lims(axs)
for ax in axs[1:]:
    ax.set_yticks([])


res = xr.concat(res, dim=pd.Index(PLOT_YRS, name="year"))
for ax in axs[1:]:
    ax.plot(res.sigma, res["psi_eval"].isel(year=0), ls="--", c="k", lw=2)

plt.show()

#### Plot tendency

In [ ]:
## resample
X = Th_rolling.resample({"time": "QS-JAN"}).mean()

## subset for specific month
t_idx = dict(time=X.time.dt.month == 1)
X_ = X.sel(t_idx)

## plot years
PLOT_YRS = [1870, 2010, 2080]


fig, axs = plt.subplots(1, 3, figsize=(7.5, 2.5), layout="constrained")

for ax, yr in zip(axs, PLOT_YRS):

    ax.scatter(X_["ddt_eli_05_scaled"].sel(year=yr), X_["ddt_T_3"].sel(year=yr), s=10)

src.utils.set_lims(axs)
for ax in axs:
    ax.set_xlim([-7, 7])
for ax in axs[1:]:
    ax.set_yticks([])


plt.show()

### Update ELI computation

In [ ]:
def get_files(varname):
    """get files for given variable name"""

    ## path to cesm2 data in MMLEA archive
    cesm2_fp = pathlib.Path("/glade/campaign/collections/rda/data/d651039/cesm2_lens")

    ## check if ocean or atmosphere
    is_oc = varname in ["mlotst", "sos", "tos", "z20", "zos"]

    if is_oc:
        data_fp = cesm2_fp / pathlib.Path("Omon", varname)

    else:
        data_fp = cesm2_fp / pathlib.Path("Amon", varname)

    return sorted(data_fp.glob("*.nc"))


def load_member_varname(member_idx, varname):
    """load ensemble member. Args:
    - member_idx: integer in [0,99]
    """

    ## Get list of files
    files = get_files(varname)

    ## open data
    data = xr.open_dataset(files[member_idx])

    ## remove un-needed coords
    data = data[varname].squeeze(drop=True)

    ## rename lon/lat
    data = data.rename({"lat": "latitude", "lon": "longitude"})

    return data


# def get_eli(sst, sst_trop_avg):
#     """compute ELI from tropical SST data"""

#     ## get SST in tropical Pac
#     sst_pac = rsst.sel(longitude=slice(120, 285), latitude=slice(-5, 5))

#     ## get relative SST
#     rsst_pac = sst_pac - sst_trop_avg

#     ## get boolean array where SST exceeds thresh
#     exceeds_thresh = rsst_pac >= 0

#     ## sum and count longitudes exceeding thresh
#     longitude_sum = (exceeds_thresh * rsst_pac["longitude"]).sum(
#         ["longitude", "latitude"]
#     )
#     longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

#     ## eli is average longitude
#     eli0 = longitude_sum / longitude_count

#     ## now integrate a second time

#     return eli


def load_member_eli(member_idx):
    """Load eli data for given member index"""

    ## load sst data
    sst = load_member_varname(member_idx, "tos")

    ## compute sst averaged over various lat bands
    eli = []
    bands = np.arange(5, 35, 5)
    for b in bands:

        ## get sst subset
        sst_band = sst.sel(latitude=slice(-b, b))

        ## get ELI
        eli_ = get_eli(sst_band)
        eli.append(eli_.rename(f"eli_{b:02d}"))

    return xr.merge(eli)


def load_ensemble_eli(save_fp):
    """Load all ensemble members"""

    ## check if file exists
    if save_fp.is_file():

        data = xr.open_dataset(save_fp)

    else:

        ## new dimension: ensemble member
        member_dim = pd.Index(np.arange(100), name="member")

        ## do computation
        data = xr.concat(
            [load_member_eli(i) for i in tqdm.tqdm(member_dim)],
            dim=member_dim,
        )

        ## save to file
        data.to_netcdf(save_fp)

    return data

Load SST data

In [ ]:
## spatial SST
forced, anom = src.utils.load_consolidated()
sst_total = xr.merge([forced["sst"] + anom["sst"], forced["sst_comp"]])

## load tropical SST avg
Ttrop = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/trop_sst.nc"))

Preprocessing funcs

In [ ]:
def get_clim(x, n=2):
    """function to get climatology"""

    ## args
    kwargs = dict(fn=np.nanmean, n=n, min_periods=1)
    
    return src.utils.get_rolling_fn_bymonth(x, **kwargs)

def get_clim_cyc(x, n=2):
    """get climatological annual cycle"""
    
    ## get monthly clim
    clim = get_clim(x, n=n)
    
    ## get annual clim
    clim_ann = clim.rolling({"time":12}, center=True, min_periods=1).mean()
    # clim_ann = clim.groupby("time.year").mean()

    ## get seasonal cycle
    clim_cyc = clim-clim_ann
    # clim_cyc = clim.groupby("time.year") - clim_ann

    return clim_cyc

def get_clim_nocyc(x, n=2):
    """get climatology without annual cycle"""
    
    ## get monthly clim
    clim = get_clim(x, n=n)
    
    ## get annual clim
    clim_ann = clim.rolling({"time":12}, center=True).mean()
    # clim_ann = clim.groupby("time.year").mean()
    
    ## broadcast it
    clim_nocyc = clim_ann
    # clim_nocyc = clim_ann - xr.zeros_like(clim).groupby("time.year")

    return clim_nocyc

def remove_clim_cyc(x, n=2):
    """remove climatological cycle"""

    ## first, get clim. cycle
    clim_cyc = get_clim_cyc(x, n=n)

    return x - clim_cyc

In [ ]:
## number of years to calc rolling average over
# n_years = 5
# n = int((n_years - 1) / 2)
n = 2

## remove annual cycle from SST
sst_total_nocyc = remove_clim_cyc(sst_total["sst"], n=n)

## remove forced component
sst_nocyc = sst_total_nocyc - get_clim(sst_total_nocyc, n=n)

## get annual cycle (for testing)
sst_cyc = get_clim_cyc(sst_total["sst"], n=n)

## get climatology (w/o annual cycle)
Ttrop_clim_nocyc = get_clim_nocyc(Ttrop, n=n)

In [ ]:
## Add back components
add_comps = lambda x : xr.merge([x,forced["sst_comp"]])
sst_total_nocyc = add_comps(sst_total_nocyc)
sst_nocyc = add_comps(sst_nocyc)
sst_cyc = add_comps(sst_cyc)

check it makes sense

In [ ]:
T_3_cyc = src.utils.reconstruct_wrapper(
    sst_cyc,
    fn=src.utils.get_nino3,
)["sst"]

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(T_3_cyc[:480])
plt.show()

### now test func on small dataset

In [ ]:
def get_eli_helper(exceeds_thresh):
    """compute ELI from mask of threshold exceedance"""

    ## sum and count longitudes exceeding thresh
    lon = exceeds_thresh.longitude
    longitude_sum = (exceeds_thresh * lon).sum(
        ["longitude", "latitude"]
    )
    longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

    ## eli is average longitude
    eli = longitude_sum / longitude_count

    return eli

def get_eli(sst, sst_trop_avg):
    """compute ELI from tropical SST data"""

    ## reconstruct equatorial SST
    sst_pac = src.utils.reconstruct_wrapper(
        sst, fn=lambda x: x.sel(longitude=slice(120, 280), latitude=slice(-5, 5))
        # sst, fn=lambda x: x.sel(longitude=slice(140, 285), latitude=slice(-5, 5))
    )

    ## get relative SST
    rsst_pac = sst_pac - sst_trop_avg

    ## get boolean array where SST exceeds thresh
    exceeds_thresh0 = rsst_pac >= 1.5

    ## compute initial ELI estimate
    eli0 = get_eli_helper(exceeds_thresh0)

    ## fill in everything west of ELI
    lon = exceeds_thresh0.longitude
    exceeds_thresh1 = exceeds_thresh0 | (lon<=eli0)

    ## integrate again
    eli1 = get_eli_helper(exceeds_thresh1)

    return eli1["sst"], exceeds_thresh1

Compute

In [ ]:
## specify time indices
# idx = dict(time=slice(None, 1200))
idx = dict(time=slice(None, None))

## get reconstructed data
x = xr.merge([sst_total_nocyc, forced["sst_comp"]])
y = xr.merge([sst_nocyc, forced["sst_comp"]])

T_3 = []
eli0 = []
members = sst_total.member[:10]
for member in tqdm.tqdm(members):

    ## get index
    idx = dict(member=member)

    ## compute Niño 3 index
    T_3.append(
        src.utils.reconstruct_wrapper(
            sst_nocyc.sel(idx), fn=src.utils.get_nino3
        )["sst"]
    )

    ## compute ELI
    eli0.append(
        get_eli(sst_total_nocyc.sel(idx), Ttrop_clim_nocyc["trop_sst_05"])[0]
    )

## concat lists
T_3 = xr.concat(T_3, dim=members)
eli0 = xr.concat(eli0, dim=members)

Plot (compare to Niño 3)

In [ ]:
idx = dict(member=1, time=slice(None,120))

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(eli0.isel(idx), lw=2)
# ax.plot(eli["eli_05"].isel(member=0, **idx))

ax2 = ax.twinx()
ax2.plot(T_3.isel(idx), c="gray", zorder=0.1, ls="--")
# ax2.plot(Th["T_3"].isel(member=0, **idx), c="k")

plt.show()

In [ ]:
## plot ELI0 vs. ELI1
idx = dict(time=slice(0,480,12))
sc_kwargs = dict(s=10)

fig, ax = plt.subplots(figsize=(3,3), layout="constrained")
ax.scatter(
    eli_all["eli_05"].isel(member=members, **idx), 
    eli0.isel(**idx), 
    **sc_kwargs,
)

ax.plot(np.linspace(150,200),np.linspace(150,200), c="k")
ax.set_aspect("equal")
plt.show()


## plot ELI vs. T3
fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")
axs[0].scatter(eli0.isel(idx), T_3.isel(idx), **sc_kwargs)
axs[1].scatter( eli_all["eli_05"].isel(member=members, **idx), T_3.isel(idx), **sc_kwargs)